# Evaluation

This example evaluates a model on held out students. It covers:

1. Splitting by student.
2. `accuracy`, `rmse` and `auc` per KC.
3. Point estimate versus posterior predictions, and why smoothed predictions should not be used for evaluation.

See the [Evaluation Metrics](../guide/metrics.rst) guide page for details.

In [1]:
import logging

import numpy as np
import pandas as pd
from cmdstanpy.utils import get_logger

# keep the notebook output readable, CmdStan progress is logged at INFO level
get_logger().setLevel(logging.WARNING)

In [2]:
from stanbkt.utils import sim_simple_BKT

data_df = sim_simple_BKT(n_students=200, n_problems=20, n_kcs=3, rng_seed=71,
                         learn=[0.05, 0.15, 0.3], guess=[0.25, 0.2, 0.15], slip=0.1)

## Split by student

Hold out whole students, not individual interactions, so the test set measures how well the model predicts students it has never seen.

In [3]:
rng = np.random.default_rng(0)
students = data_df["student_id"].unique()
test_students = rng.choice(students, size=len(students) // 5, replace=False)
train_df = data_df[~data_df["student_id"].isin(test_students)]
test_df = data_df[data_df["student_id"].isin(test_students)]
print(train_df["student_id"].nunique(), "train students,", test_df["student_id"].nunique(), "test students")

160 train students, 40 test students


In [4]:
from stanbkt.fits import MCMCFitOptions
from stanbkt.models import StandardBKT
from stanbkt.utils import VerbosityLevel

model = StandardBKT(verbose=VerbosityLevel.WARN)
model.fit(train_df, stan_fit_options=MCMCFitOptions(seed=1234, iter_warmup=500, iter_sampling=500, show_progress=False))

23:41:51 - cmdstanpy - WARNING - Some chains may have failed to converge.
	Chain 1 had 1 divergent transitions (0.2%)
	Use the "diagnose()" method on the CmdStanMCMC object to see further information.


StandardBKT(fit_method=<FitMethod.MCMC: 'mcmc'>, verbose=<VerbosityLevel.WARN: 1>, is_fitted=True)

## Metrics per KC

Use the filtered predictions from `predict`: `pCorrectness` at each interaction only uses the responses before it, so it is a genuine prediction. Each KC is a separate model, so score them separately.

In [5]:
from stanbkt.utils import accuracy, auc, rmse


def score(preds, column="pCorrectness"):
    rows = []
    for kc, kc_preds in preds.groupby("kc_id", observed=True):
        y, p = kc_preds["correct"], kc_preds[column]
        rows.append({"kc_id": kc, "accuracy": accuracy(y, p), "rmse": rmse(y, p), "auc": auc(y, p)})
    return pd.DataFrame(rows).set_index("kc_id")


point_preds = model.predict(test_df)
score(point_preds).round(4)

,accuracy,rmse,auc
kc_id,,,
kc_0,0.6438,0.4821,0.5940
kc_1,0.7056,0.4469,0.7451
kc_2,0.6917,0.4598,0.6696


`auc` matches `sklearn.metrics.roc_auc_score`, including tied predictions. Ties are common in BKT: with population initial knowledge every student's first interaction has the same `pCorrectness`.

## Point estimate versus posterior mean

`predict_posterior_summary` averages the prediction over the posterior instead of plugging in one value per parameter.

In [6]:
posterior = model.predict_posterior_summary(test_df, backend="numba")
pd.concat({"point estimate": score(point_preds), "posterior mean": score(posterior, "pCorrectness_mean")}, axis=1).round(4)

point estimate                 posterior mean                
            accuracy    rmse     auc       accuracy    rmse     auc
kc_id                                                              
kc_0          0.6438  0.4821  0.5940         0.6469  0.4820  0.5935
kc_1          0.7056  0.4469  0.7451         0.7056  0.4469  0.7450
kc_2          0.6917  0.4598  0.6696         0.6917  0.4597  0.6696

With enough data the two agree closely. The posterior mean matters more when the parameters are uncertain, for example in KCs with few students.

## Do not evaluate with smoothed predictions

Smoothed predictions condition on the whole sequence, including the response they are predicting, so their scores are optimistic and not comparable with other models.

In [7]:
smoothed = model.predict_smoothed(test_df)
pd.concat({"filtered": score(point_preds), "smoothed (leaks the answer)": score(smoothed)}, axis=1).round(4)

filtered                 smoothed (leaks the answer)                
      accuracy    rmse     auc                    accuracy    rmse     auc
kc_id                                                                     
kc_0    0.6438  0.4821  0.5940                      0.7844  0.3960  0.9200
kc_1    0.7056  0.4469  0.7451                      0.8722  0.3337  0.9328
kc_2    0.6917  0.4598  0.6696                      0.9417  0.2486  0.9836